# 19 Bolshoi halos from CosmoSim

Halo positions for Bolshoi (250 h⁻¹ Mpc, 2048³ particles, WMAP5), so that the tracer-based void finders of notebook 18 (VIDE, VoidFinder) can run on it, plus a density grid and T-web labels that line up with them. Run all; nothing needs to be run by hand. Downloads and results are kept on Drive and reused.

1. **Download** the most massive `N_KEEP` BDM halos (Bound Density Maximum, overdensity 360 × background) at z = 0 (snapshot 416) from CosmoSim's `bolshoi.bdmv` table, in one job. λ = 3 h⁻¹ Mpc needs (250/3)³ ≈ 579,000 halos.
2. **Reading the old density grid.** The repo's Bolshoi grids (`bolshoi/dm`, `bolshoi/halos/L{s}`, 640³) were, as far as Sam remembers, made on 624³ cells and padded to 640³ by copying the first cells onto the high ends. Section 2 checks for that padding, then correlates the halos with the grid read both ways (all 640³ cells as 0.391 h⁻¹ Mpc cells, or the first 624³ as 0.401 h⁻¹ Mpc cells) and with both grid-point placements, and keeps the reading that lines up.
3. **Density and labels** on the grid that lines up. For the 624³ reading: the dark-matter density cropped to 624³ and T-web labels computed from it (σ_eff = 1 h⁻¹ Mpc, threshold 0.65), saved for notebook 18.
4. **Which halos the old halo grids used** (hosts and subhalos or hosts only; weighted by mass or not), read the same way. Results go to `data/Bolshoi/bolshoi_halo_check.json`, which notebook 18 reads.

If nothing lines up, the diagnostics at the end (sections 2b–2e) run and report what they find; they do not stop the run. No GPU needed.

**Token.** As notebook 15: the CosmoSim API token in the Colab secret `COSMOSIM_TOKEN`; never written into the notebook.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install pyvo
!rm -rf /content/deepvoid
!git clone -q "https://github.com/skumagai3/deepvoid.git" /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1

In [ ]:
import dataclasses
import itertools
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvo
import requests

from deepvoid import Catalog
from deepvoid.grid import cic
from deepvoid.observe import select_by_spacing
from deepvoid.resample import block_mean
from deepvoid.sources.bolshoi import BOX_SIZE, SNAPSHOT_Z0, read_halos

catalog = Catalog()
DATA = catalog.root / 'Bolshoi'
HALO_FILE = DATA / 'bolshoi_bdmv_z0_top2M.parquet'
CHECK_FILE = DATA / 'bolshoi_halo_check.json'
N_KEEP = 2_000_000
# Diagnostics for a failed alignment (sections 2b-2e). Each has run once; its results are in the notes.
# Set one to True to repeat it. Downloads and the 128^3 comparison grids are kept on Drive and reused.
# Diagnostics, run only if section 2 finds no reading that lines up (each one is cached or quick to repeat)
RUN = {'2b scale/flip/axes': True, '2c translation': True, '2d CosmoSim density': True, '2e Bolshoi-Planck': True}
ALIGN_FILE = DATA / 'bolshoi_alignment_v2.json'  # section 2's result, reused on reruns
FIELDS_FILE = DATA / 'bolshoi_fields128.npz'     # the 128^3 grids compared in 2d and 2e
SL = slice(60, 64)                               # slab for the figures: 4 cells of 1.95 h^-1 Mpc in x


def xcorr(a, b):
    # correlation of standardized a and b for every periodic translation of a (in cells)
    fa, fb = np.fft.rfftn(a), np.fft.rfftn(b)
    return np.fft.irfftn(fa * np.conj(fb), s=a.shape, axes=(0, 1, 2)) / a.size


def standard(g):
    g = np.log1p(g / g.mean())
    return ((g - g.mean()) / g.std()).astype(np.float32)
SPACINGS = [3, 5, 7, 10]
TAP_URL = 'https://www.cosmosim.org/tap'
QUEUE = '1h'

session = requests.Session()
session.headers['Authorization'] = f"Token {userdata.get('COSMOSIM_TOKEN')}"
tap = pyvo.dal.TAPService(TAP_URL, session=session)
langs = []
for lang in tap.get_tap_capability().languages:
    versions = [v.content for v in lang.versions if v.content] if lang.versions else []
    langs += [f'{lang.name}-{v}' for v in versions] or [lang.name]
LANGUAGE = next((l for l in langs if l.lower().startswith('postgres')), None)
assert LANGUAGE, f'no PostgreSQL language listed: {langs}'
print('using', LANGUAGE)

COLUMNS = 'bdmid, hostflag, x, y, z, vx, vy, vz, mvir'
N_OLD, PAD = 640, 16                             # the repo's old Bolshoi grids; padding to test for
N_REAL = N_OLD - PAD
DENSITY_FILE = DATA / 'bolshoi_dm_n624.npz'      # section 3, for the 624^3 reading
LABEL_FILE = DATA / 'bolshoi_mask_tweb_th0.65_seff1_n624.npz'

## 1. Download

One job on the 1-hour queue: the `N_KEEP` most massive z = 0 halos. Skipped if the file is already on Drive.

In [ ]:
if HALO_FILE.exists():
    print(HALO_FILE.name, 'already there')
else:
    t0 = time.time()
    query = f'SELECT {COLUMNS} FROM bolshoi.bdmv WHERE snapnum = {SNAPSHOT_Z0} ORDER BY mvir DESC LIMIT {N_KEEP}'
    job = tap.submit_job(query, language=LANGUAGE, queue=QUEUE, runid='bolshoi_bdmv_z0_top')
    job.run()
    while job.phase not in ('COMPLETED', 'ERROR', 'ABORTED'):
        print(f'{job.phase} after {time.time() - t0:.0f} s', flush=True)
        time.sleep(30)
    if job.phase != 'COMPLETED':
        job.raise_if_error()
        raise RuntimeError(f'job ended as {job.phase}')
    df = job.fetch_result().to_table().to_pandas()
    print(f'{len(df):,} rows in {time.time() - t0:.0f} s')
    assert len(df) == N_KEEP, 'fewer halos than asked for: check the query'
    df = df.astype({'x': 'float32', 'y': 'float32', 'z': 'float32',
                    'vx': 'float32', 'vy': 'float32', 'vz': 'float32', 'mvir': 'float32'})
    DATA.mkdir(parents=True, exist_ok=True)
    df.to_parquet(HALO_FILE, index=False)
    job.delete()
    del df

halos = read_halos(HALO_FILE)
hosts = read_halos(HALO_FILE, distinct_only=True)
print(f'{len(halos):,} halos, {halos.meta["n_subhalos"]:,} of them subhalos; '
      f'mvir {halos.mass.min():.3g} to {halos.mass.max():.3g} h^-1 Msun; '
      f'positions {halos.pos.min():.2f} to {halos.pos.max():.2f} h^-1 Mpc')
for s in SPACINGS:
    need = round((BOX_SIZE / s) ** 3)
    print(f'λ = {s}: needs {need:,} halos; mass cut {np.sort(halos.mass)[-need]:.3g} (all), '
          f'{np.sort(hosts.mass)[-need]:.3g} (hosts only)')
assert len(hosts) >= round((BOX_SIZE / min(SPACINGS)) ** 3)

## 2. Reading the old density grid

**Padding.** Along each axis: are the last 16 slices of `bolshoi/dm` identical to the first 16? An unrelated slab is shown for comparison. If all three axes are padded, the real grid is the first 624³ cells.

**Alignment.** The λ = 3 halos gridded on the same cells, correlated (log(1 + ρ/ρ̄), whole box) with the density for each reading and each grid-point placement (point *i* at *i* × cell or at (*i* + ½) × cell). A reading lines up if its correlation is above 0.3 (CosmoSim's own Bolshoi density gave 0.67 with these halos on 128³ cells; random placement gives about 0).

In [ ]:
from deepvoid.labels import tweb_labels
from deepvoid.io import load_volume, save_volume
from deepvoid.volume import Volume

n = N_OLD


def grid(tracers, shift, size, weights='mass'):
    # point i at (i + shift) * cell: move the tracers by -shift cells, then cic (point i at i * cell)
    c = BOX_SIZE / size
    moved = dataclasses.replace(tracers, pos=(tracers.pos - shift * c) % BOX_SIZE)
    g = cic(moved, size, weights=weights).data
    return g / g.mean()


if ALIGN_FILE.exists():
    saved = json.loads(ALIGN_FILE.read_text())
    padding, align = pd.DataFrame(saved['padding']), pd.DataFrame(saved['rows'])
    print('section 2 from', ALIGN_FILE.name, '(delete it to recompute)')
else:
    dm = catalog.load('bolshoi/n640-padded/dm').data.astype(np.float32)
    assert dm.shape[0] == N_OLD
    rows = []
    for ax in range(3):
        for lo, what in ((N_REAL, 'last 16 vs first 16'), (N_OLD // 2, 'unrelated: middle 16 vs first 16')):
            a = np.take(dm, range(lo, lo + PAD), axis=ax).ravel()
            b = np.take(dm, range(PAD), axis=ax).ravel()
            rows.append({'axis': 'xyz'[ax], 'slabs': what, 'identical': bool(np.array_equal(a, b)),
                         'corr': round(float(np.corrcoef(np.log1p(a), np.log1p(b))[0, 1]), 4)})
    padding = pd.DataFrame(rows)
    padded = bool(padding[padding['slabs'] == 'last 16 vs first 16']['identical'].all())
    readings = {N_OLD: dm}
    if padded:
        readings[N_REAL] = np.ascontiguousarray(dm[:N_REAL, :N_REAL, :N_REAL])
    sample = select_by_spacing(halos, 3)
    rows = []
    for size, field in readings.items():
        d = standard(field).ravel()
        for shift in (0.0, 0.5):
            g = standard(grid(sample, shift, size, weights=None)).ravel()
            rows.append({'cells': size, 'reading': f'all {N_OLD}^3 cells' if size == N_OLD else f'first {N_REAL}^3 cells (unpadded)',
                         'shift': shift, 'corr': float(np.mean(g * d))})
            del g
        del d
    align = pd.DataFrame(rows)
    del dm, readings, sample
    ALIGN_FILE.write_text(json.dumps({'padding': padding.to_dict('records'), 'rows': align.to_dict('records')}))
print(padding.to_string(index=False))
align = align.sort_values('corr', ascending=False)
print(align.round(4).to_string(index=False))
best = align.iloc[0]
ALIGNED = bool(best['corr'] > 0.3)
N_GRID, GRID_SHIFT = int(best['cells']), float(best['shift'])
cell = BOX_SIZE / N_GRID
if ALIGNED:
    print(f'lines up: {best["reading"]}, point i at (i + {GRID_SHIFT}) cells, correlation {best["corr"]:.3f}')
else:
    print('NOTHING LINES UP (best correlation', round(float(best['corr']), 3), '): the diagnostics at the end run; '
          'nothing is saved for notebook 18')

## 3. Density and labels on the grid that lines up

For the 624³ reading: the dark-matter density cropped to its first 624³ cells and the T-web labels computed from it with the rule used everywhere else (σ_eff = 1 h⁻¹ Mpc, so σ = √(1 − 0.401²) ≈ 0.92 h⁻¹ Mpc of Gaussian smoothing, which is the old mask's `sig=0.916`; threshold 0.65), both saved for notebook 18. For the 640³ reading the repo's `bolshoi/dm` and `bolshoi/mask` are used as they are. Skipped if the files are already there.

In [ ]:
if ALIGNED and N_GRID == N_REAL:
    if not (DENSITY_FILE.exists() and LABEL_FILE.exists()):
        t0 = time.time()
        dm = catalog.load('bolshoi/n640-padded/dm').data[:N_REAL, :N_REAL, :N_REAL].astype(np.float32)
        rho = Volume(data=dm / dm.mean(), box_size=BOX_SIZE, meta={'field': 'density', 'sim': 'bolshoi',
                     'note': 'first 624^3 cells of bolshoi/dm (a 624^3 grid padded to 640^3)'})
        del dm
        labels = tweb_labels(rho, sigma_eff=1.0, threshold=0.65)
        save_volume(rho, DENSITY_FILE, dtype='float32')
        save_volume(labels, LABEL_FILE)
        print(f'saved {DENSITY_FILE.name} and {LABEL_FILE.name} in {time.time() - t0:.0f} s')
        del rho, labels
    lab = load_volume(LABEL_FILE, dtype=None).data
    print('class shares (void, wall, filament, halo):', np.round(np.bincount(lab.ravel(), minlength=4) / lab.size, 3))
    old = catalog.load('bolshoi/n640-padded/mask').data[:N_REAL, :N_REAL, :N_REAL]
    print(f'agreement with the repo\'s 640^3 labels cropped to 624^3: {np.mean(old == lab):.3f} of voxels '
          '(those were computed on the padded grid as if its cells were 0.391 h^-1 Mpc)')
    del lab, old
elif ALIGNED:
    print('640^3 reading: notebook 18 uses bolshoi/dm and bolshoi/mask as they are')
else:
    print('skipped: nothing lines up')

## 4. Which halos the old halo grids used

For each old halo grid `bolshoi/halos/L{s}` (read the same way as the density) and each of our selections, the correlation of log(1 + ρ/ρ̄) at the placement found in section 2. Also the ratio of the old grid's total to the number of halos and to their total mass, which hints at whether it was weighted by mass. Then the results for notebook 18 are saved.

In [ ]:
if ALIGNED:
    rows = []
    for s in SPACINGS:
        old = catalog.load(f'bolshoi/n640-padded/halos/L{s}').data[:N_GRID, :N_GRID, :N_GRID].astype(np.float32)
        o = standard(old).ravel()
        for label, tr in (('all', halos), ('hosts only', hosts)):
            sel = select_by_spacing(tr, s)
            for weights in ('mass', None):
                rows.append({'λ': s, 'halos': label, 'weights': weights or 'count',
                             'corr': float(np.mean(standard(grid(sel, GRID_SHIFT, N_GRID, weights)).ravel() * o)),
                             'old total / N': float(old.sum()) / len(sel.mass),
                             'old total / mass': float(old.sum()) / float(sel.mass.sum())})
        del old, o
    match = pd.DataFrame(rows)
    print(match.round(4).to_string(index=False))
    best_sel = match.groupby(['halos', 'weights'])['corr'].mean().idxmax()
    print('best on average:', best_sel)
    check = {'aligned': True, 'n': N_GRID, 'grid_shift': GRID_SHIFT, 'alignment_corr': float(best['corr']),
             'distinct_only': best_sel[0] == 'hosts only', 'weights': best_sel[1], 'halo_file': HALO_FILE.name,
             'density_file': DENSITY_FILE.name if N_GRID == N_REAL else None,
             'label_file': LABEL_FILE.name if N_GRID == N_REAL else None,
             'padding': padding.to_dict('records'), 'alignment': align.to_dict('records'),
             'old_grid_match': match.to_dict('records')}
    CHECK_FILE.write_text(json.dumps(check, indent=1))
    print('saved', CHECK_FILE)
else:
    CHECK_FILE.write_text(json.dumps({'aligned': False, 'alignment': align.to_dict('records')}, indent=1))
    print('nothing lines up; saved that to', CHECK_FILE.name, 'so notebook 18 will not use Bolshoi')

## Diagnostics (only if section 2 found nothing that lines up)

Sections 2b–2e were written while looking for the cause (scale, flips and axis order; a translation; CosmoSim's own Bolshoi density; Bolshoi-Planck). They run only when section 2 fails, each switched in `RUN`, and an error in one is printed and skipped so Run all finishes.

### 2b Looking for the cause (only if section 2 failed and its switch in `RUN` is on)

If the halos do not follow the density, something maps their positions differently from the grid. On coarse 128³ grids (fast), this tries every combination of: a scale on the positions (1; 1/h and h for Mpc against h⁻¹ Mpc; 1/1000 for kpc), flipping each axis, and the 6 axis orders, and lists the best. The positions as stored are summarized first. A combination well above the rest (correlation of 0.5 or more) is the answer.

In [ ]:
if not ALIGNED and RUN['2b scale/flip/axes']:
    try:
        p = halos.pos
        print('positions: min', p.min(0).round(3), 'max', p.max(0).round(3), 'mean', p.mean(0).round(3))
        raw = pd.read_parquet(HALO_FILE, columns=['x', 'y', 'z'])
        print('raw x, y, z before wrapping: min', raw.min().round(3).tolist(), 'max', raw.max().round(3).tolist())
        del raw
        nc = min(128, n)
        dmc = block_mean(catalog.load('bolshoi/n640-padded/dm').data.astype(np.float32), n // nc)
        lc = np.log1p(dmc / dmc.mean()).ravel(); lc = (lc - lc.mean()) / lc.std()
        del dmc
        sample = select_by_spacing(halos, 3)
        rows = []
        for scale in (1.0, 1 / 0.70, 0.70, 1e-3):
            moved = dataclasses.replace(sample, pos=(sample.pos * scale) % BOX_SIZE)
            g = cic(moved, nc, weights=None).data
            g = np.log1p(g / g.mean())
            for flips in itertools.product((False, True), repeat=3):
                gf = np.flip(g, [a for a in range(3) if flips[a]]) if any(flips) else g
                for perm in itertools.permutations(range(3)):
                    v = np.transpose(gf, perm).ravel()
                    rows.append({'scale': round(scale, 4), 'flip': ''.join('xyz'[a] for a in range(3) if flips[a]) or '-',
                                 'axes': ''.join('xyz'[q] for q in perm),
                                 'corr': float(np.mean((v - v.mean()) / v.std() * lc))})
        search = pd.DataFrame(rows).sort_values('corr', ascending=False)
        print(search.head(12).to_string(index=False))
    except Exception as err:
        print('2b failed, going on:', repr(err))

### 2c Is the box shifted? (only if section 2 failed and its switch in `RUN` is on)

Section 2b only tries scales, flips and axis orders. A constant translation between the halo positions and the grid (a different origin) would also break the alignment and leave a weak large-scale correlation. The cross-correlation by FFT gives the correlation for every translation at once: for each flip and axis order on 128³ grids, the best translation and its correlation; then the best case again on 320³ for a finer translation. A peak of 0.5 or more at a nonzero translation means the positions are shifted by that much.

In [ ]:
if not ALIGNED and RUN['2c translation']:
    try:
        sample = select_by_spacing(halos, 3)
        dm_full = catalog.load('bolshoi/n640-padded/dm').data.astype(np.float32)
        rows = []
        for nc in (min(128, n),):
            b = standard(block_mean(dm_full, n // nc))
            g = standard(cic(sample, nc, weights=None).data)
            for flips in itertools.product((False, True), repeat=3):
                gf = np.flip(g, [a for a in range(3) if flips[a]]) if any(flips) else g
                for perm in itertools.permutations(range(3)):
                    c = xcorr(np.ascontiguousarray(np.transpose(gf, perm)), b)
                    k = np.unravel_index(np.argmax(c), c.shape)
                    shift = np.where(np.array(k) > nc // 2, np.array(k) - nc, np.array(k)) * (BOX_SIZE / nc)
                    rows.append({'flip': ''.join('xyz'[a] for a in range(3) if flips[a]) or '-',
                                 'axes': ''.join('xyz'[q] for q in perm), 'peak corr': float(c.max()),
                                 'corr at 0': float(c[0, 0, 0]), 'translation (h^-1 Mpc)': np.round(shift, 2).tolist()})
        xs = pd.DataFrame(rows).sort_values('peak corr', ascending=False)
        print(xs.head(8).to_string(index=False))
        top = xs.iloc[0]
        flips = [ 'xyz'.index(a) for a in top['flip'] if a != '-']
        perm = ['xyz'.index(a) for a in top['axes']]
        nf = n // 2 if n % 2 == 0 else n
        b = standard(block_mean(dm_full, n // nf))
        g = standard(cic(sample, nf, weights=None).data)
        g = np.ascontiguousarray(np.transpose(np.flip(g, flips) if flips else g, perm))
        c = xcorr(g, b)
        k = np.array(np.unravel_index(np.argmax(c), c.shape))
        k = np.where(k > nf // 2, k - nf, k)
        print(f'{nf}^3, flip {top["flip"]}, axes {top["axes"]}: peak correlation {c.max():.4f} at a translation of '
              f'{k} cells = {np.round(k * BOX_SIZE / nf, 3)} h^-1 Mpc (the halos sit this far from the matching structure in the grid; subtracting it from their positions aligns them)')
        del dm_full, b, g, c
    except Exception as err:
        print('2c failed, going on:', repr(err))

### 2d Test 1: CosmoSim's own Bolshoi density at z = 0 (only if section 2 failed and its switch in `RUN` is on)

CosmoSim serves Bolshoi's dark-matter density at z = 0 on a 256³ grid (`bolshoi.dens256_z0`; its schema page calls it strongly smoothed, which is fine for a comparison on 1.95 h⁻¹ Mpc cells). The table with all snapshots (`bolshoi.dens256`) timed out when filtered by snapshot. It is downloaded in 4 chunks of 64 x-slices (a query returns at most 5,000,000 rows) and kept on Drive. Then three fields are compared on a common 128³ grid (1.95 h⁻¹ Mpc cells): the old density grid (`bolshoi/dm`), CosmoSim's density, and the λ = 3 halos.

- If the halos follow CosmoSim's density, the halos are right and the old grid differs.
- If CosmoSim's density also differs from the old grid, the old grid is not Bolshoi at z = 0 as CosmoSim serves it (test 2 checks Bolshoi-Planck).

For each pair: the correlation with no shift and the best over all periodic shifts. Then the three fields side by side on the same slab.

In [ ]:
if not ALIGNED and not FIELDS_FILE.exists():
    try:
        # CosmoSim's density (downloaded once) and the 128^3 grids compared below, kept on Drive
        if (DATA / 'bolshoi_dens256_z0.npy').exists():
            cs = np.load(DATA / 'bolshoi_dens256_z0.npy')
        else:
            D256 = DATA / 'dens256_z0'
            D256.mkdir(exist_ok=True)
            TABLE = 'bolshoi.dens256_z0'
            probe = tap.run_sync(f'SELECT * FROM {TABLE} LIMIT 3', language=LANGUAGE).to_table().to_pandas()
            print(probe)
            cols = {c.lower(): c for c in probe.columns}
            DENS = next(c for c in ('dens', 'density', 'delta', 'rho') if c in cols)
            assert all(c in cols for c in ('ix', 'iy', 'iz')), f'no ix, iy, iz columns: {list(probe.columns)}'
            for a in range(0, 256, 64):
                path = D256 / f'dens256_ix{a:03d}.parquet'
                if path.exists():
                    continue
                t0 = time.time()
                upper = '' if a == 192 else f' AND ix < {a + 64}'       # the last chunk also takes ix = 256 if indices start at 1
                q = f'SELECT ix, iy, iz, {cols[DENS]} AS dens FROM {TABLE} WHERE ix >= {a}{upper}'
                job = tap.submit_job(q, language=LANGUAGE, queue=QUEUE, runid=f'bolshoi_dens256_ix{a:03d}')
                job.run()
                job.wait(phases=['COMPLETED', 'ERROR', 'ABORTED'], timeout=3600)
                if job.phase != 'COMPLETED':
                    job.raise_if_error()
                    raise RuntimeError(f'{path.name}: job ended as {job.phase}')
                df = job.fetch_result().to_table().to_pandas()
                df.columns = [c.lower() for c in df.columns]
                df.astype({'ix': 'int16', 'iy': 'int16', 'iz': 'int16', 'dens': 'float32'}).to_parquet(path, index=False)
                job.delete()
                print(f'{path.name}: {len(df):,} rows in {time.time() - t0:.0f} s', flush=True)
            df = pd.concat([pd.read_parquet(p) for p in sorted(D256.glob('dens256_ix*.parquet'))])
            lo = int(df[['ix', 'iy', 'iz']].min().min())
            print(f'{len(df):,} rows; indices start at {lo}; dens from {df.dens.min():.3g} to {df.dens.max():.3g}, mean {df.dens.mean():.3g}')
            assert len(df) == 256 ** 3, 'not every cell once'
            cs = np.zeros((256, 256, 256), np.float32)
            cs[df.ix.to_numpy() - lo, df.iy.to_numpy() - lo, df.iz.to_numpy() - lo] = df.dens.to_numpy()
            del df
            if cs.min() < 0:                       # an overdensity delta rather than rho / rho_bar
                cs = cs + 1
            np.save(DATA / 'bolshoi_dens256_z0.npy', cs)
        old = {k: catalog.load(name).data for k, name in (('old_dm', 'bolshoi/n640-padded/dm'), ('old_L3', 'bolshoi/n640-padded/halos/L3'),
                                                         ('old_L5', 'bolshoi/n640-padded/halos/L5'))}
        assert all(v.shape[0] == n for v in old.values())
        np.savez(FIELDS_FILE, cosmosim=block_mean(cs, 2), bolshoi_halos_L3=cic(select_by_spacing(halos, 3), 128, weights=None).data,
                 **{k: block_mean(v.astype(np.float32), n // 128) for k, v in old.items()})
        del cs, old
    except Exception as err:
        print('2d failed, going on:', repr(err))

if not ALIGNED:
    try:
        with np.load(FIELDS_FILE) as f:
            F128 = {k: f[k] for k in f.files}
    except Exception as err:
        print('2d failed, going on:', repr(err))

if not ALIGNED and RUN['2d CosmoSim density']:
    try:
        fields = {'old grid (bolshoi/dm)': F128['old_dm'], 'CosmoSim dens256, z = 0': F128['cosmosim'],
                  'halos, λ = 3 (CosmoSim BDM)': F128['bolshoi_halos_L3']}
        std = {k: standard(v) for k, v in fields.items()}
        rows = []
        for a, b in itertools.combinations(std, 2):
            c = xcorr(std[a], std[b])
            k = np.array(np.unravel_index(np.argmax(c), c.shape)); k = np.where(k > 64, k - 128, k)
            rows.append({'pair': f'{a}  vs  {b}', 'corr, no shift': round(float(c[0, 0, 0]), 4),
                         'best corr': round(float(c.max()), 4), 'at shift (h^-1 Mpc)': np.round(k * BOX_SIZE / 128, 2).tolist()})
        print(pd.DataFrame(rows).to_string(index=False))

        fig, axes = plt.subplots(1, 3, figsize=(16, 5.6))
        for ax, (name, v) in zip(axes, fields.items()):
            img = np.log10(1 + v[SL].mean(0) / v.mean())
            im = ax.imshow(img.T, origin='lower', extent=(0, BOX_SIZE, 0, BOX_SIZE), cmap='magma',
                           vmin=np.percentile(img, 1), vmax=np.percentile(img, 99.5))
            ax.set_title(name, fontsize=10)
            ax.set_xlabel('y (h⁻¹ Mpc)'); ax.set_ylabel('z (h⁻¹ Mpc)')
            fig.colorbar(im, ax=ax, shrink=0.8, label='log₁₀(1 + ρ/ρ̄), slab mean')
        fig.suptitle(f'Bolshoi, slab x = {60 * BOX_SIZE / 128:.0f}–{64 * BOX_SIZE / 128:.0f} h⁻¹ Mpc, on 128³ cells: '
                     'the old density grid, CosmoSim\'s z = 0 density, and the halos')
        fig.tight_layout()
        fig.savefig(DATA / 'bolshoi_alignment_slices.png', dpi=130, bbox_inches='tight')
        plt.show()
    except Exception as err:
        print('2d failed, going on:', repr(err))

### 2e Test 2: are the old grids Bolshoi-Planck? (only if section 2 failed and its switch in `RUN` is on)

Bolshoi-Planck (CosmoSim schema `bolshoip`) is a 250 h⁻¹ Mpc, 2048³ run with Planck cosmology. Its BDM halos at z = 0 (the snapshot with the lowest redshift in `bolshoip.redshifts`) are downloaded the same way as Bolshoi's, then compared on 128³ cells with the old density grid, the old halo grids at λ = 3 and 5, and CosmoSim's Bolshoi density from 2d. If the old grids are Bolshoi-Planck, the Bolshoi-Planck halos follow them closely with no shift, and the old halo grids nearly match (correlation near 1, as on fake data).

In [ ]:
if not ALIGNED and RUN['2e Bolshoi-Planck']:
    try:
        zs = tap.run_sync('SELECT * FROM bolshoip.redshifts', language=LANGUAGE).to_table().to_pandas()
        zs.columns = [c.lower() for c in zs.columns]
        zcol = next(c for c in ('zred', 'redshift', 'z') if c in zs.columns)
        zs = zs.sort_values(zcol)
        print(zs.head(3).to_string(index=False))
        SNAP_P = int(zs.iloc[0]['snapnum'])
        print('Bolshoi-Planck z = 0 snapshot:', SNAP_P)
        BP_FILE = DATA / 'bolshoip_bdmv_z0_top2M.parquet'
        if not BP_FILE.exists():
            t0 = time.time()
            q = f'SELECT {COLUMNS} FROM bolshoip.bdmv WHERE snapnum = {SNAP_P} ORDER BY mvir DESC LIMIT {N_KEEP}'
            job = tap.submit_job(q, language=LANGUAGE, queue=QUEUE, runid='bolshoip_bdmv_z0_top')
            job.run()
            while job.phase not in ('COMPLETED', 'ERROR', 'ABORTED'):
                print(f'{job.phase} after {time.time() - t0:.0f} s', flush=True)
                time.sleep(30)
            if job.phase != 'COMPLETED':
                job.raise_if_error()
                raise RuntimeError(f'job ended as {job.phase}')
            df = job.fetch_result().to_table().to_pandas()
            df.columns = [c.lower() for c in df.columns]
            df = df.astype({c: 'float32' for c in ('x', 'y', 'z', 'vx', 'vy', 'vz', 'mvir')})
            df.to_parquet(BP_FILE, index=False)
            job.delete()
            print(f'{len(df):,} Bolshoi-Planck halos in {time.time() - t0:.0f} s')
            del df
        bp = read_halos(BP_FILE)               # Bolshoi's reader: same columns, same 250 h^-1 Mpc box
        print(f'{len(bp):,} halos; mvir {bp.mass.min():.3g} to {bp.mass.max():.3g}; positions {bp.pos.min():.2f} to {bp.pos.max():.2f}')

        old = {'old grid (bolshoi/dm)': F128['old_dm'], 'old halo grid L3': F128['old_L3'],
               'old halo grid L5': F128['old_L5'], 'CosmoSim Bolshoi dens256, z = 0': F128['cosmosim']}
        mine = {'Bolshoi-Planck halos, λ = 3': cic(select_by_spacing(bp, 3), 128, weights=None).data,
                'Bolshoi-Planck halos, λ = 5': cic(select_by_spacing(bp, 5), 128, weights=None).data,
                'Bolshoi halos, λ = 3': F128['bolshoi_halos_L3']}
        rows = []
        for a, va in mine.items():
            sa = standard(va)
            for b, vb in old.items():
                c = xcorr(sa, standard(vb))
                k = np.array(np.unravel_index(np.argmax(c), c.shape)); k = np.where(k > 64, k - 128, k)
                rows.append({'halos': a, 'grid': b, 'corr, no shift': round(float(c[0, 0, 0]), 4),
                             'best corr': round(float(c.max()), 4), 'at shift (h^-1 Mpc)': np.round(k * BOX_SIZE / 128, 2).tolist()})
        print(pd.DataFrame(rows).to_string(index=False))

        show = {'old grid (bolshoi/dm)': old['old grid (bolshoi/dm)'], 'Bolshoi-Planck halos, λ = 3': mine['Bolshoi-Planck halos, λ = 3'],
                'CosmoSim Bolshoi dens256, z = 0': old['CosmoSim Bolshoi dens256, z = 0'], 'Bolshoi halos, λ = 3': mine['Bolshoi halos, λ = 3']}
        fig, axes = plt.subplots(1, 4, figsize=(21, 5.6))
        for ax, (name, v) in zip(axes, show.items()):
            img = np.log10(1 + v[SL].mean(0) / v.mean())
            im = ax.imshow(img.T, origin='lower', extent=(0, BOX_SIZE, 0, BOX_SIZE), cmap='magma',
                           vmin=np.percentile(img, 1), vmax=np.percentile(img, 99.5))
            ax.set_title(name, fontsize=10)
            ax.set_xlabel('y (h⁻¹ Mpc)'); ax.set_ylabel('z (h⁻¹ Mpc)')
            fig.colorbar(im, ax=ax, shrink=0.8, label='log₁₀(1 + ρ/ρ̄), slab mean')
        fig.suptitle(f'Same slab (x = {60 * BOX_SIZE / 128:.0f}–{64 * BOX_SIZE / 128:.0f} h⁻¹ Mpc, 128³ cells): '
                     'old grid and Bolshoi-Planck halos (left pair), CosmoSim Bolshoi density and Bolshoi halos (right pair)')
        fig.tight_layout()
        fig.savefig(DATA / 'bolshoi_planck_test.png', dpi=130, bbox_inches='tight')
        plt.show()
    except Exception as err:
        print('2e failed, going on:', repr(err))

## Shut down

Deletes this runtime once every cell above has finished, so it stops using compute units. The results are already on Drive; anything kept only under `/content` goes with the runtime. With **Run all**, Colab stops at the first cell that fails, so after an error the runtime stays up and you can look at it. Set `SHUT_DOWN = False` to keep the runtime.

In [ ]:
SHUT_DOWN = True   # False keeps the runtime after the run
if SHUT_DOWN:
    import time
    from google.colab import drive, runtime
    drive.flush_and_unmount()   # finish writing to Drive first
    time.sleep(5)
    runtime.unassign()          # disconnects and deletes this runtime